# Part 1 — Imports

In [1]:
from pathlib import Path
import torch

from dmpbridge.pdf.page_image_converter import convert_pdf_to_images
from dmpbridge.vision.qwen_structure_detector import detect_structure_from_images
from dmpbridge.vision.qwen_postprocessor import save_qwen_structured_blocks
from dmpbridge.processing.structure_json_builder import save_narrative_json
from dmpbridge.pdf.pdfplumber_extractor import save_pdfplumber_outputs
from dmpbridge.processing.structure_detector import detect_structure

# Part 2 — Check GPU

In [2]:
import torch

print("Torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())

for i in range(torch.cuda.device_count()):
    print(i, torch.cuda.get_device_name(i))

Torch version: 2.6.0+cu124
CUDA available: True
GPU count: 1
0 NVIDIA GeForce RTX 3090


# Part 3 — Paths

In [3]:
project_root = Path.cwd().parent

pdf_path = project_root / "data" / "raw_pdfs" / "sample6.pdf"
skeleton_path = project_root / "schemas" / "rda_dmp_dmptool_extension_skeleton.json"

qwen_output_path = project_root / "data" / "qwen_outputs" / f"{pdf_path.stem}.json"
qwen_structured_path = project_root / "data" / "qwen_outputs" / f"{pdf_path.stem}_structured_blocks.json"
qwen_json_path = project_root / "data" / "structure_json" / f"{pdf_path.stem}_qwen.json"

print("Project root:", project_root)
print("PDF path:", pdf_path)
print("PDF exists:", pdf_path.exists())
print("Skeleton exists:", skeleton_path.exists())

Project root: c:\Users\Nahid\dmpbridge
PDF path: c:\Users\Nahid\dmpbridge\data\raw_pdfs\sample6.pdf
PDF exists: True
Skeleton exists: True


# Part 4 — Convert PDF to page images

In [ ]:
image_paths = convert_pdf_to_images(pdf_path, dpi=120)

print("Number of page images:", len(image_paths))

for p in image_paths:
    print(p, p.exists())

[2026-05-08 10:27:57] Converting PDF pages to images: sample6.pdf
[2026-05-08 10:27:57] Saved 2 page images to: C:\Users\Nahid\dmpbridge\data\page_images\sample6
Number of page images: 2
C:\Users\Nahid\dmpbridge\data\page_images\sample6\page_1.png True
C:\Users\Nahid\dmpbridge\data\page_images\sample6\page_2.png True


# Part 5 — Run Qwen2-VL structure detection

In [6]:
qwen_results = detect_structure_from_images(
    image_paths=image_paths,
    output_path=qwen_output_path
)

print("Saved Qwen output:", qwen_output_path.exists())
print("Qwen output path:", qwen_output_path)

qwen_results

[2026-05-08 10:28:24] Loading Qwen2-VL model: Qwen/Qwen2-VL-7B-Instruct


Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/730 [00:00<?, ?it/s]

[2026-05-08 10:28:55] Running Qwen2-VL on page 1: C:\Users\Nahid\dmpbridge\data\page_images\sample6\page_1.png


KeyboardInterrupt: 

# Part 6 — Print Qwen output clearly

In [ ]:
for page in qwen_results:
    print("\nPAGE:", page.get("page"))

    if "error" in page:
        print("ERROR:", page["error"])
        print(page.get("raw_response", "")[:1000])

    for item in page.get("items", []):
        print(item.get("label"), "→", item.get("text"))

# Part 7 — Convert Qwen output to structured blocks

In [ ]:
blocks = save_pdfplumber_outputs(pdf_path)
structured_blocks = detect_structure(blocks)

print("Rule-based structural labels:")

for block in structured_blocks:
    if block["label"] in ["document_title", "section", "subsection", "question"]:
        print(block["label"], "→", block["text"])

# Part 8 — Convert Qwen output to structured blocks

In [ ]:
qwen_structured_blocks = save_qwen_structured_blocks(
    qwen_output_path=qwen_output_path,
    output_path=qwen_structured_path,
    source_pdf=pdf_path.name
)

print("Saved Qwen structured blocks:", qwen_structured_path.exists())
print("Number of Qwen structured blocks:", len(qwen_structured_blocks))

qwen_structured_blocks[:10]

# Part 9 — Build narrative JSON from Qwen blocks

In [ ]:
qwen_json = save_narrative_json(
    structured_blocks=qwen_structured_blocks,
    output_path=qwen_json_path,
    skeleton_path=skeleton_path
)

print("Saved Qwen narrative JSON:", qwen_json_path.exists())
print("Qwen JSON path:", qwen_json_path)

# Part 10 — Inspect Qwen narrative JSON

In [ ]:
sections = qwen_json["narrative"]["template"]["section"]

print("Number of sections:", len(sections))

for section in sections:
    print(section["order"], section["title"], "| questions:", len(section["question"]))

In [ ]:
sections[0] if sections else "No sections created"